# Benchmark your own synchronisation algorithm

The bench runs estimators on simulated networks where the true clock offset is known, and scores
them against that truth. This notebook adds a new estimator, a few lines of Python, and compares
it with the reference algorithms that ship with ntpstats: a Kalman filter with RTS smoother, the
NTP clock filter, chrony-style regression and a RADclock-style feed-forward estimator.

<sub>MIT licensed, part of [ntpstats](https://github.com/thiagodefreitas/NetworkTime). Runs in CI on every change.</sub>

In [ ]:
from pathlib import Path

import numpy as np

from ntpstats import api as nt

# examples/data, whether the notebook runs from examples/notebooks or the repository root
DATA = next(p for p in (Path("../data"), Path("examples/data"), Path("../../examples/data")) if p.is_dir())

print("built-in estimators:", ", ".join(nt.available_estimators()))

## The estimator

An estimator takes the measured series (offset and delay per sample) and returns its estimate
of the true offset at each sample. This one averages only the samples whose delay is close to the
minimum, since queueing makes the others biased.

In [ ]:
class LowDelayEWMA(nt.Estimator):
    name = "low-delay-ewma"
    description = "EWMA of samples within 1.5x the minimum delay"

    def __init__(self, alpha=0.3, factor=1.5):
        self.alpha, self.factor = alpha, factor

    def fit(self, s: nt.TimeSeries) -> nt.TimeSeries:
        d = s.extra["delay"]
        good = d <= self.factor * np.min(d)
        est, out = s.offset[0], np.empty(len(s))
        for i, (x, g) in enumerate(zip(s.offset, good)):
            if g:
                est += self.alpha * (x - est)
            out[i] = est
        return nt.TimeSeries(s.t, out, name=f"{s.name} [{self.name}]")


nt.register_estimator(LowDelayEWMA())

## Scenarios and scores

Scenarios are presets (`lan`, `internet`, `congested`, `route-change`, `falseticker`) or TOML files.
Each runs with several seeds; the score is the error against the true offset after a warm-up.

In [ ]:
from ntpstats.bench import summarize

scenarios = nt.load_scenarios(["lan", "congested", "route-change"])
rows = nt.run_bench(scenarios, estimators=["raw", "mindelay", "regression", "kalman", "low-delay-ewma"],
                    seeds=(1, 2), duration=6 * 3600)
table = summarize(rows)
cols = ["scenario", "estimator", "rms", "p95_abs", "max_abs"]
print(" ".join(f"{c:>16}" for c in cols))
for row in table:
    print(" ".join(f"{row[c]:>16.3e}" if isinstance(row[c], float) else f"{row[c]:>16}" for c in cols))

Read the table per scenario, best first. The toy estimator beats the raw samples when queues are
long (`congested`), because it ignores samples that waited in a queue. On a quiet LAN, where
almost every sample is good, it loses: the EWMA lags the clock's wander. That is the kind of
trade-off the bench is for: try `alpha` and `factor`, or add your own scenario as a TOML file.

## A single run up close

`score` compares any estimate with the truth; `compare` adds stability of the error.

In [ ]:
queue = nt.PathModel(queue_mean=2e-3)
meas, truth = nt.simulate_ntp(nt.Scenario(duration=6 * 3600, seed=7, forward=queue, backward=queue))
mine = nt.run_estimator("low-delay-ewma", meas)
kal = nt.run_estimator("kalman", meas)
for label, est in (("raw", meas), ("low-delay-ewma", mine), ("kalman", kal)):
    sc = nt.score(est, truth, warmup=1800)
    print(f"{label:>15}: RMS error {nt.format_seconds(sc['rms'])}, max {nt.format_seconds(sc['max_abs'])}")

To share an estimator, publish it as a package with an `ntpstats.estimators` entry point; it then
appears in `ntpstats bench` and in `ntpstats plugins` (see *Writing a plugin* in the docs).